In [1]:
import os
import pandas as pd
import json
import ast

from ollama import Client, generate
import requests, json, time
from typing import Any, Dict, List, TypedDict
from langgraph.graph import StateGraph, START, END


In [2]:
df = pd.read_excel('1000_annotated_sample_final.xlsx')
df = df.drop(columns=(['Unnamed: 0', 'flag']))
df.shape

(1000, 12)

In [3]:
# df['len_manual'] = df['ABSA_manual'].apply(lambda x: len(ast.literal_eval(x).get('aspects', [])) if isinstance(x, str) else len(x.get('aspects', [])))
# df['len_atepc'] = df['atepc_json'].apply(lambda x: len(ast.literal_eval(x).get('aspect', [])) if isinstance(x, str) else len(x.get('aspect', [])))

In [4]:
df.head(2)

,place_id,author,rating,date,text,place_name,index,date_parsed,ABSA_manual,atepc_json,len_manual,len_atepc
0,ChIJs_LE-WM71moR_c251OO23DM,Edward Russel,1,3 years ago,"Utterly useless team, overpromises and underde...",Crave Cafe & Restaurant,2720,2022-10-11,"{""overall_sentiment"":""negative"",""aspects"":[{""a...","{'sentence': ""Utterly useless team , overpromi...",1,1
1,ChIJ7yq6lYXGsGoRXmQH2ld5juI,James Martin,1,Edited 5 years ago,Very very poor i would not recomend anyone use...,Australia Post - Royal Park LPO,29442,2020-10-11,"{""overall_sentiment"":""negative"",""aspects"":[{""a...",{'sentence': 'Very very poor i would not recom...,1,1


In [5]:
# df[abs(df.len_manual - df.len_atepc) > 2]

# Zero shot

## phi4:14b 

### Pipeline prompts
- aspect term extraction (ATE), which identifies the features or aspects being discussed (e.g., "food"), 
- opinion term extraction (OTE), which finds the words expressing an opinion (e.g., "delicious"), 
- aspect-level sentiment classification (ASC)

In [9]:
import dspy
import json
from typing import List, Dict, Any

In [10]:
MODEL = "phi4:14b"
API_KEY = ""
API_BASE = "http://localhost:11434"
# MODEL = "llama3.2:1b"

In [11]:
lm = dspy.LM(f"ollama_chat/{MODEL}", api_base=API_BASE, api_key=API_KEY)
dspy.configure(lm=lm)

In [27]:
class ATE_Do(dspy.Signature):
    """
    You are an expert annotator for Aspect Term Extraction (ATE), and your task is to extract explicit aspect terms mentioned in the review (≤2 words each).
    Output must be a pure array of strings.
    Example: ["aspect 1", "aspect 2", ...]
    """
    review = dspy.InputField()
    aspects = dspy.OutputField()

class ATE_Supervise(dspy.Signature):
    """
    You are the supervisor for the ATE task. Critically evaluate the proposed aspect list.
    Rules:
    - Only aspects explicitly mentioned in the text.
    - Keep each aspect to ≤2 words.
    - Remove duplicates and near-duplicates.
    Respond with JSON:
      {"decision": "accept" or "revise",
       "revised_aspects": [...], 
       "rationale": "short reason"}
    """
    review = dspy.InputField()
    proposed_aspects = dspy.InputField()
    # critique_history = dspy.InputField()
    decision_json = dspy.OutputField()

In [28]:
class OTE_Do(dspy.Signature):
    """
    You are an expert annotator for Opinion Term Extraction (OTE). For each aspect, extract one short evidence phrase (≤3 words, verbatim).
    Output JSON array of objects:
        [{"aspect":..., "opinion":...}, ...]
    """
    review = dspy.InputField()
    aspects = dspy.InputField()
    opinions = dspy.OutputField()

class OTE_Supervise(dspy.Signature):
    """
    You are the supervisor for the OTE task. Check evidence phrases are short (≤3 words) and quoted verbatim.
    Keep one best phrase per aspect. Respond with JSON:
      {"decision": "accept" or "revise",
       "revised_opinions":[{"aspect":..., "opinion":...},...],
       "rationale": "short reason"}
    """
    review = dspy.InputField()
    # aspects = dspy.InputField()
    proposed_opinions = dspy.InputField()
    # critique_history = dspy.InputField()
    decision_json = dspy.OutputField()

In [53]:
class ALSC_Do(dspy.Signature):
    """
    You are an expert annotator for Aspect-level Sentiment Classification (ALSC). For each aspect, classify sentiment as 'positive' or 'negative' or 'neutral'.
    Only use explicit evidence. Output JSON array:
        [{"aspect":..., "opinion":..., "sentiment": 'positive' or 'negative' or 'neutral'}, ...]
    """
    review = dspy.InputField()
    aspects = dspy.InputField()
    sentiments = dspy.OutputField()

class ALSC_Supervise(dspy.Signature):
    """
    You are the supervisor for the ALSC task.. Verify labels align with explicit evidence (not assumptions).
    Respond with JSON:
      {"decision": "accept" or "revise",
       "sentiments": ["aspect":..., "opinion":..., "sentiment": 'positive' or 'negative' or 'neutral'}, ...],
       "rationale": ...}
    """
    review = dspy.InputField()
    aspects = dspy.InputField()
    proposed_sentiments = dspy.InputField()
    # critique_history = dspy.InputField()
    decision_json = dspy.OutputField()

In [46]:
class Overall_Do(dspy.Signature):
    """
    Overall sentiment for the whole review: 'positive' or 'negative' or 'neutral'.
    """
    review = dspy.InputField()
    overall = dspy.OutputField()

class Overall_Supervise(dspy.Signature):
    """
    Supervisor: Ensure the overall sentiment reflects the full tone (not just the average).
    Respond with JSON: {"decision": "accept" or "revise", "overall":"...", "rationale":"..."}
    """
    review = dspy.InputField()
    proposed_overall = dspy.InputField()
    # critique_history = dspy.InputField()
    decision_json = dspy.OutputField()

In [47]:
def _safe_json_loads(s, default):
    try:
        return json.loads(s) if isinstance(s, str) else s
    except Exception:
        return default

In [48]:
class DebateStep(dspy.Module):
    """
    Generic two-agent debate: Doer proposes, Supervisor critiques/edits.
    Stops early on accept or after max_rounds; always returns the supervisor's latest JSON.
    """
    def __init__(self, do_sig, sup_sig, field_names: Dict[str,str], max_rounds: int = 3):
        super().__init__()
        self.doer = dspy.Predict(do_sig)
        self.supervisor = dspy.Predict(sup_sig)
        self.max_rounds = max_rounds
        self.field_names = field_names
        # field_names = {
        # "proposal_arg": "proposed_aspects",  # what arg name the Supervisor expects
        # "payload": "aspects",                # key to extract from supervisor output
        # "supervisor_out": "decision_json"    # field name in the supervisor’s output
        # }

    def forward(self, **kwargs) -> Dict[str, Any]:
        history = []
        proposal_key = self.field_names["proposal_arg"]  # arg name passed to supervisor
        payload_field = self.field_names["payload"]      # key inside supervisor JSON

        # Round 0: doer proposes
        do_out = self.doer(**kwargs)
        proposal = getattr(do_out, payload_field, None)  # ✅ read the correct attr
        if proposal is None:
            proposal = getattr(do_out, proposal_key, None)  # fallback
        
        proposal_text = proposal if isinstance(proposal, str) else json.dumps(proposal, ensure_ascii=False)

        for _ in range(self.max_rounds):
            sup = self.supervisor(**{**kwargs,
                                     self.field_names["proposal_arg"]: proposal_text,
                                     "critique_history": json.dumps(history, ensure_ascii=False)})
            sup_json = _safe_json_loads(getattr(sup, self.field_names["supervisor_out"]), {})
            decision = (sup_json.get("decision", "") or "").lower()
            payload = sup_json.get(payload_field, None)
            rationale = sup_json.get("rationale", "")
            history.append({"proposal": proposal_text, "decision": decision, "rationale": rationale})

            if decision == "accept" and payload is not None:
                return {"decision": "accept", payload_field: payload, "history": history}

            # On revise, feed back the revised payload as next proposal
            if payload is not None:
                proposal_text = json.dumps(payload, ensure_ascii=False)
            else:
                # If supervisor didn't return a valid payload, keep previous
                pass

        # If we exit without accept, return last payload we have
        return {"decision": "finalize", payload_field: _safe_json_loads(proposal_text, None), "history": history}

In [49]:
class ABSAState(TypedDict):
    review: str
    aspects: List[str]
    opinions: List[Dict[str, str]]
    sentiments: List[Dict[str, str]]
    overall_sentiment: str
    # _hist_ate: List[Dict[str, Any]]
    # _hist_ote: List[Dict[str, Any]]
    # _hist_alsc: List[Dict[str, Any]]
    # _hist_overall: List[Dict[str, Any]]

In [50]:
class ABSALangGraph:
    def __init__(self, max_rounds: int = 3):
        self.ate = DebateStep(
            ATE_Do, ATE_Supervise,
            field_names=dict(proposal_arg="proposed_aspects", payload="aspects", supervisor_out="decision_json"),
            max_rounds=max_rounds
        )
        self.ote = DebateStep(
            OTE_Do, OTE_Supervise,
            field_names=dict(proposal_arg="proposed_opinions", payload="opinions", supervisor_out="decision_json"),
            max_rounds=max_rounds
        )
        self.alsc = DebateStep(
            ALSC_Do, ALSC_Supervise,
            field_names=dict(proposal_arg="proposed_sentiments", payload="sentiments", supervisor_out="decision_json"),
            max_rounds=max_rounds
        )
        self.overall = DebateStep(
            Overall_Do, Overall_Supervise,
            field_names=dict(proposal_arg="proposed_overall", payload="overall", supervisor_out="decision_json"),
            max_rounds=max_rounds
        )

        sg = StateGraph(ABSAState)

        def node_ate(state: ABSAState) -> ABSAState:
            out = self.ate(review=state["review"])
            aspects = _safe_json_loads(out.get("aspects"), [])
            state["aspects"] = aspects or []
            # state["_hist_ate"] = out.get("history", [])
            return state

        def node_ote(state: ABSAState) -> ABSAState:
            aspects = state.get("aspects", [])
            if not aspects:
                state["opinions"] = []
                # state["_hist_ote"] = []
                return state
            out = self.ote(review=state["review"], aspects=json.dumps(aspects, ensure_ascii=False))
            opinions = _safe_json_loads(out.get("opinions"), [])
            op_map = {}
            for o in opinions:
                if isinstance(o, dict) and "aspect" in o:
                    a = o["aspect"]
                    if a not in op_map and isinstance(o.get("opinion",""), str):
                        op_map[a] = o["opinion"]
            state["opinions"] = [{"aspect": a, "opinion": op_map.get(a, "")} for a in aspects]
            # state["_hist_ote"] = out.get("history", [])
            return state

        def node_alsc(state: ABSAState) -> ABSAState:
            aspects = state.get("aspects", [])
            if not aspects:
                state["sentiments"] = []
                # state["_hist_alsc"] = []
                return state
            out = self.alsc(review=state["review"], aspects=json.dumps(aspects, ensure_ascii=False))
            sentiments = _safe_json_loads(out.get("sentiments"), [])
            sent_map = {}
            for s in sentiments:
                if isinstance(s, dict) and "aspect" in s:
                    sent_map[s["aspect"]] = s.get("label","neutral")
            state["sentiments"] = [{"aspect": a, "label": sent_map.get(a, "neutral")} for a in aspects]
            # state["_hist_alsc"] = out.get("history", [])
            return state

        def node_overall(state: ABSAState) -> ABSAState:
            out = self.overall(review=state["review"], proposed_overall="neutral")
            overall = (out.get("overall") or "neutral").strip().lower()
            if overall not in {"positive","negative","neutral"}:
                overall = "neutral"
            state["overall_sentiment"] = overall
            # state["_hist_overall"] = out.get("history", [])
            return state

        sg.add_node("ATE", node_ate)
        sg.add_node("OTE", node_ote)
        sg.add_node("ALSC", node_alsc)
        sg.add_node("OVERALL", node_overall)

        sg.add_edge(START, "ATE")
        sg.add_edge("ATE", "OTE")
        sg.add_edge("OTE", "ALSC")
        sg.add_edge("ALSC", "OVERALL")
        sg.add_edge("OVERALL", END)

        self.app = sg.compile()

    def run(self, review: str, include_history: bool = False) -> Dict[str, Any]:
        init: ABSAState = {"review": review}
        final_state: ABSAState = self.app.invoke(init)
        out = {
            "aspects": final_state.get("aspects", []),
            "opinions": final_state.get("opinions", []),
            "sentiments": final_state.get("sentiments", []),
            "overall_sentiment": final_state.get("overall_sentiment", "neutral"),
        }
        # if include_history:
        #     out["_histories"] = {
        #         "ate": final_state.get("_hist_ate", []),
        #         "ote": final_state.get("_hist_ote", []),
        #         "alsc": final_state.get("_hist_alsc", []),
        #         "overall": final_state.get("_hist_overall", []),
        #     }
        return out

In [51]:
def process_dataframe(df: pd.DataFrame, text_col: str, max_rounds: int) -> pd.DataFrame:
    runner = ABSALangGraph(max_rounds=max_rounds)
    results = []
    for txt in df[text_col].astype(str).fillna(""):
        results.append(json.dumps(runner.run(txt), ensure_ascii=False))
    out = df.copy()
    out["absa_json"] = results
    return out

In [56]:
runner

In [52]:
if __name__ == "__main__":
    runner = ABSALangGraph(max_rounds=2)

    texts = [
        "The food was excellent, but the service was painfully slow.",
    ]
    for t in texts:
        print("\nREVIEW:", t)
        out = runner.run(t)
        print(json.dumps(out, indent=2, ensure_ascii=False))



REVIEW: The food was excellent, but the service was painfully slow.
{
  "aspects": [
    "food",
    "service"
  ],
  "opinions": [
    {
      "aspect": "food",
      "opinion": "excellent"
    },
    {
      "aspect": "service",
      "opinion": "painfully slow"
    }
  ],
  "sentiments": [
    {
      "aspect": "food",
      "label": "neutral"
    },
    {
      "aspect": "service",
      "label": "neutral"
    }
  ],
  "overall_sentiment": "neutral"
}


In [15]:
# class ABSAWithDebate(dspy.Module):
#     def __init__(self, max_rounds=3):
#         super().__init__()
#         self.ate = DebateStep(
#             ATE_Do, 
#             ATE_Supervise,
#             field_names=dict(proposal_arg="proposed_aspects", payload="aspects", supervisor_out="decision_json"),
#             max_rounds=max_rounds
#         )
#         self.ote = DebateStep(
#             OTE_Do, 
#             OTE_Supervise,
#             field_names=dict(proposal_arg="proposed_opinions", payload="opinions", supervisor_out="decision_json"),
#             max_rounds=max_rounds
#         )
#         self.alsc = DebateStep(
#             ALSC_Do, 
#             ALSC_Supervise,
#             field_names=dict(proposal_arg="proposed_sentiments", payload="sentiments", supervisor_out="decision_json"),
#             max_rounds=max_rounds
#         )
#         self.overall = DebateStep(
#             Overall_Do, 
#             Overall_Supervise,
#             field_names=dict(proposal_arg="proposed_overall", payload="overall", supervisor_out="decision_json"),
#             max_rounds=max_rounds
#         )

#     def forward(self, review: str) -> Dict[str, Any]:
#         # ---- ATE debate ----
#         ate = self.ate(review=review)
#         aspects = ate.get("aspects") or []
#         aspects = _safe_json_loads(aspects, [])  # ensure list

#         # If nothing extracted, short-circuit with neutrals
#         if not aspects:
#             ov = self.overall(review=review, proposed_overall="neutral")
#             overall = ov.get("overall") or "neutral"
#             return {"aspects": [], "opinions": [], "sentiments": [], "overall_sentiment": overall}

#         # ---- OTE debate ----
#         ote = self.ote(review=review, aspects=json.dumps(aspects, ensure_ascii=False))
#         opinions = _safe_json_loads(ote.get("opinions"), [])
#         # normalize one opinion per aspect
#         op_map = {}
#         for o in opinions:
#             if isinstance(o, dict) and "aspect" in o:
#                 a = o["aspect"]
#                 if a not in op_map and isinstance(o.get("opinion",""), str):
#                     op_map[a] = o["opinion"]
#         opinions_out = [{"aspect": a, "opinion": op_map.get(a, "")} for a in aspects]

#         # ---- ALSC debate ----
#         alsc = self.alsc(review=review, aspects=json.dumps(aspects, ensure_ascii=False))
#         sentiments = _safe_json_loads(alsc.get("sentiments"), [])
#         sent_map = {}
#         for s in sentiments:
#             if isinstance(s, dict) and "aspect" in s:
#                 sent_map[s["aspect"]] = s.get("label","neutral")
#         sentiments_out = [{"aspect": a, "label": sent_map.get(a, "neutral")} for a in aspects]

#         # ---- Overall debate ----
#         ov = self.overall(review=review, proposed_overall="neutral")
#         overall = ov.get("overall") or "neutral"
#         overall = overall.strip().lower()
#         if overall not in {"positive","negative","neutral"}:
#             overall = "neutral"

#         return {
#             "aspects": aspects,
#             "opinions": opinions_out,
#             "sentiments": sentiments_out,
#             "overall_sentiment": overall
#         }

In [16]:
# if __name__ == "__main__":
#     absa = ABSAWithDebate(max_rounds=3)

#     texts = [
#         "The food was excellent, but the service was painfully slow.",
#     ]
#     for t in texts:
#         print("\nREVIEW:", t)
#         out = absa(review=t)
#         print(json.dumps(out, indent=2, ensure_ascii=False))


REVIEW: The food was excellent, but the service was painfully slow.
{
  "aspects": [
    "food",
    "service"
  ],
  "opinions": [
    {
      "aspect": "food",
      "opinion": "excellent"
    },
    {
      "aspect": "service",
      "opinion": "painfully slow"
    }
  ],
  "sentiments": [
    {
      "aspect": "food",
      "label": "positive"
    },
    {
      "aspect": "service",
      "label": "negative"
    }
  ],
  "overall_sentiment": "neutral"
}


In [16]:
# MODEL = "phi4:14b"
MODEL = "llama3.2:1b"

In [17]:
# Schemas
ate_schema = """
{
  "aspects": [
    {
      "term": "string",            // ≤ 2 words, lemmatized if plural
      "category": "string"    // optional coarse tag (e.g., service, price)
    }
  ]
}"""
ote_schema = """
{
  "opinions": [
    {
      "aspect": string,           // aspect from ATE aspects[]
      "opinion": "string",         // ≤ 3 words
      "link_type": "explicit | implicit"
    }
  ]
}
"""
alsc_schema = """
{
  "sentiments": [
    {
      "aspect": string,           // aspect from ATE aspects[]
      "category": "string",       // category from ATE aspects[]
      "label": "positive|negative|neutral",
      "evidence_span": "string",   // short verbatim quote (≤ 6 words)
    }
  ],
  "overall_sentiment": "positive|negative|neutral"
}
"""

In [39]:
def ate(review_text:str, ate_schema:str):
    ate_prompt = f"""
    System:
    You are an expert annotator for Aspect Term Extraction (ATE). Extract explicit aspects as short noun phrases (≤ 2 words). Return valid JSON exactly matching the provided schema. Use character offsets for the original text.

    User:
    TEXT:
    {review_text}

    SCHEMA:
    {ate_schema}

    GUIDELINES:
    - Only include aspects that are explicitly mentioned (no inferred entities).
    - Prefer concrete entities over vague words (e.g., “customer service” over “experience”).
    - Merge duplicates; do not overlap spans.
    - Output ONLY the JSON and nothing else.
    """
    return generate(model=MODEL, 
                        prompt=ate_prompt,
                        )

In [ ]:
def ote(review_text:str, ote_schema:str, ate_out:str):
    aspects_json = ate_out
    ote_prompt = f"""
    System:
    You are an expert annotator for Opinion Term Extraction (OTE). For each given aspect, extract the shortest opinion expression (≤ 3 words) directly supporting an attitude toward that aspect. Return JSON per the schema.

    User:
    TEXT:
    {review_text}

    ASPECTS (index-aligned):
    {aspects_json}

    SCHEMA:
    {ote_schema}

    GUIDELINES:
    - Link opinions to aspects.
    - “explicit” if the opinion directly modifies the aspect; otherwise “implicit”.
    - If no opinion exists for an aspect, omit it.
    - Output ONLY the JSON and nothing else.
    """
    return generate(model=MODEL, 
                        prompt=ote_prompt,
                        )

In [ ]:
def alsc(review_text:str, alsc_schema:str, ate_out:str, ote_out:str):
    aspects_json = ate_out
    opinions_json = ote_out
    alsc_prompt = f"""
    System:
    You are an expert for Aspect-Level Sentiment Classification (ALSC). For each aspect, assign sentiment using the text and extracted opinions. Return valid JSON.

    User:
    TEXT:
    {review_text}

    ASPECTS:
    {aspects_json}

    OPINIONS:
    {opinions_json}

    SCHEMA:
    {alsc_schema}

    GUIDELINES:
    - Use evidence_span as a short verbatim quote (≤ 4 words).
    - If multiple opinions conflict, choose the dominant one; lower confidence if ambiguous.
    - overall_sentiment is the holistic tone of the whole review.
    - Output ONLY the JSON and nothing else.
    """
    return generate(model=MODEL, 
                        prompt=alsc_prompt,
                        )

In [21]:
def absa_pipeline(review_text:str, ate_schema:str, ote_schema:str, alsc_schema:str) -> Dict[str,Any]:
    response = ate(review_text, ate_schema=ate_schema)
    ate_out = response['response']
    response = ote(review_text, ote_schema, ate_out)
    ote_out = response['response']
    response = alsc(review_text, alsc_schema, ate_out, ote_out)
    alsc_out = response['response']
    
    # aspects = ate_out.get("aspects", [])
    # sentiments = {s["aspect_index"]: s for s in alsc_out.get("sentiments", [])}
    # opinions_by_aspect: Dict[int, List[Dict[str,Any]]] = {}
    # for o in ote_out.get("opinions", []):
    #     opinions_by_aspect.setdefault(o["aspect_index"], []).append(o)

    # results = []
    # for i, asp in enumerate(aspects):
    #     results.append({
    #         "aspect": asp.get("term"),
    #         "category": asp.get("category"),
    #         "span": [asp.get("start"), asp.get("end")],
    #         "opinions": opinions_by_aspect.get(i, []),
    #         "sentiment": sentiments.get(i, {}).get("label"),
    #         "evidence_span": sentiments.get(i, {}).get("evidence_span"),
    #         "confidence": sentiments.get(i, {}).get("confidence"),
    #     })
    # return {
    #     "items": results,
    #     "overall_sentiment": alsc_out.get("overall_sentiment", "neutral")
    # }
    return alsc_out

In [22]:
review_text = df.loc[0, 'text']
review_text

'I have always been a big fan of UFC and what they did for the world of sport and martial arts in general and I was very excited to see that there is a UFC gym being built where in Castle Hill (which is where I live). However, I was less …'

In [23]:
response = ate(review_text, ate_schema=ate_schema)
ate_out = response['response']
response = ote(review_text, ote_schema, ate_out)
ote_out = response['response']
response = alsc(review_text, alsc_schema, ate_out, ote_out)
alsc_out = response['response']

In [29]:
print(alsc_out)

### Aspect-Level Sentiment Classification

#### System Overview
The system uses a combination of aspect term extraction (ATE) annotations and opinion expressions to perform sentiment classification for each aspect.

#### Input Schema:

```json
{
  "sentiments": [
    {
      "aspect": string,           // aspect from ATE aspects[]
      "category": "string",       // category from ATE aspects[]
      "label": "positive|negative|neutral",
      "evidence_span": "string",   // short verbatim quote (≤ 4 words)
    }
  ],
  "overall_sentiment": "positive|negative|neutral"
}
```

#### Guidelines:

- Use evidence_span as a short verbatim quote (≤ 6 words).
- If multiple opinions conflict, choose the dominant one; lower confidence if ambiguous.
- overall_sentiment is the holistic tone of the whole review.
- Output ONLY the JSON.

### Code Implementation

```javascript
/**
 * Aspect-Level Sentiment Classification System
 */

class ALC {
  /**
   * Constructor
   */
  constructor() {}

  /**
  

In [33]:
respons = absa_pipeline(review_text, ate_schema, ote_schema, alsc_schema)
print(response)

model='llama3.2:1b' created_at='2025-11-04T01:21:33.4990226Z' done=True done_reason='stop' total_duration=92955034600 load_duration=293894400 prompt_eval_count=698 prompt_eval_duration=2065250900 eval_count=974 eval_duration=81338425200 response='### Aspect-Level Sentiment Classification\n\n#### System Overview\nThe system uses a combination of aspect term extraction (ATE) annotations and opinion expressions to perform sentiment classification for each aspect.\n\n#### Input Schema:\n\n```json\n{\n  "sentiments": [\n    {\n      "aspect": string,           // aspect from ATE aspects[]\n      "category": "string",       // category from ATE aspects[]\n      "label": "positive|negative|neutral",\n      "evidence_span": "string",   // short verbatim quote (≤ 4 words)\n    }\n  ],\n  "overall_sentiment": "positive|negative|neutral"\n}\n```\n\n#### Guidelines:\n\n- Use evidence_span as a short verbatim quote (≤ 6 words).\n- If multiple opinions conflict, choose the dominant one; lower confid

### NLP feed
feed the result of NLP as the imput of LLM

In [30]:
def extract_absa_fields(absa: dict) -> dict:
    """
    Extracts fields from ABSA JSON output row.

    Expected keys:
      sentence, tokens, IOB, aspect, position, sentiment, probs, confidence
    """
    sample = ast.literal_eval(absa)
    aspects = sample.get("aspect", [])
    sentiments = sample.get("sentiment", [])

    return aspects, sentiments

In [32]:
def absa(review_text:str, absa_fields, alsc_schema):
    prompt = f"""
    System:
    You are an expert in Aspect-Based Sentiment Analysis (ABSA).
    You are given ABSA annotations produced by a RoBERTa model.
    Your job is to evaluate, correct, and improve these annotations — NOT just repeat them.

    Task Requirements:
    1. Validate and refine aspect terms (2 words max).
    2. Confirm or adjust sentiment: positive / negative / neutral.
    3. Correct aspect-opinion alignment if needed.
    4. Quote short evidence spans (≤ 6 words) from the text.
    5. Maintain consistent aspect indices.
    6. Only include explicit, text-grounded aspects (no hallucination).
    7. Output ONLY valid JSON per schema.

    Schema:
    {alsc_schema}

    User Input:
    TEXT:
    {review_text}

    RO_BERTA_ABSA_OUTPUT:
    {absa_fields}

    Instructions:
    - Review each aspect prediction. Correct where needed.
    - Drop incorrect aspects.
    - Add missing aspects only if explicit in the text.
    - Use short verbatim evidence from the review.
    - Output ONLY JSON. No explanations.
    """

    return generate(model=MODEL, 
                        prompt=prompt,
                        )

In [42]:
review_text = df.loc[1, 'text']
atepc_json = df.loc[1, 'atepc_json']
review_text

'Very unfriendly staff and service !'

In [44]:
aspects, sentiments = extract_absa_fields(atepc_json)
absa_fields = {'aspects': aspects, 'sentiments': sentiments}
result = absa(review_text, absa_fields, alsc_schema)
print(result['response'])

{
  "sentiments": [
    {
      "aspect": "staff",
      "category": "customer service",
      "label": "Negative",
      "evidence_span": "Very unfriendly staff and service !"
    },
    {
      "aspect": "service",
      "category": "hospitality",
      "label": "Negative"
    }
  ],
  "overall_sentiment": "Negative"
}


In [47]:
aspects

['staff', 'service']

### NLP feed + Pipeline

In [48]:
def nlp_ate(review_text:str, nlp_aspects:list, ate_schema:str):
    ate_prompt = f"""
    System:
    You are an expert annotator for Aspect Term Extraction (ATE). 
    You are given aspects for ABSA produced by a RoBERTa model.
    Your job is to evaluate, correct, and improve these aspects — NOT just repeat them.

    User:
    TEXT:
    {review_text}

    RO_BERTA_ABSA_OUTPUT:
    {nlp_aspects}

    SCHEMA:
    {ate_schema}

    GUIDELINES:
    - Only include aspects that are explicitly mentioned (no inferred entities).
    - Prefer concrete entities over vague words (e.g., “customer service” over “experience”).
    - Merge duplicates; do not overlap spans.
    - Output ONLY the JSON and nothing else.
    """
    return generate(model=MODEL, 
                        prompt=ate_prompt,
                        )

In [51]:
def nlp_ote(review_text:str, nlp_sentiments:list, ote_schema:str, ate_out:str):
    aspects_json = ate_out
    ote_prompt = f"""
    System:
    You are an expert annotator for Opinion Term Extraction (OTE). 
    You are given sentiments for ABSA produced by a RoBERTa model.
    Your job is to evaluate, correct, and improve these sentiments considering the aspects — NOT just repeat them.

    User:
    TEXT:
    {review_text}

    ASPECTS:
    {aspects_json}

    RO_BERTA_ABSA_OUTPUT:
    {nlp_sentiments}

    SCHEMA:
    {ote_schema}

    GUIDELINES:
    - Link opinions to aspects.
    - “explicit” if the opinion directly modifies the aspect; otherwise “implicit”.
    - If no opinion exists for an aspect, omit it.
    - Output ONLY the JSON and nothing else.
    """
    return generate(model=MODEL, 
                        prompt=ote_prompt,
                        )

In [52]:
def alsc(review_text:str, alsc_schema:str, ate_out:str, ote_out:str):
    aspects_json = ate_out
    opinions_json = ote_out
    alsc_prompt = f"""
    System:
    You are an expert for Aspect-Level Sentiment Classification (ALSC). For each aspect, assign sentiment using the text and extracted opinions. Return valid JSON.

    User:
    TEXT:
    {review_text}

    ASPECTS:
    {aspects_json}

    OPINIONS:
    {opinions_json}

    SCHEMA:
    {alsc_schema}

    GUIDELINES:
    - Use evidence_span as a short verbatim quote (≤ 4 words).
    - If multiple opinions conflict, choose the dominant one; lower confidence if ambiguous.
    - overall_sentiment is the holistic tone of the whole review.
    - Output ONLY the JSON and nothing else.
    """
    return generate(model=MODEL, 
                        prompt=alsc_prompt,
                        )

In [ ]:
def nlp_absa_pipeline(review_text:str, absa_fields:str, ate_schema:str, ote_schema:str, alsc_schema:str) -> Dict[str,Any]:
    nlp_aspects = absa_fields['aspects']
    nlp_sentiments = absa_fields['sentiments']
    response = nlp_ate(review_text, nlp_aspects, ate_schema)
    ate_out = response['response']
    response = nlp_ote(review_text, nlp_sentiments, ote_schema, ate_out)
    ote_out = response['response']
    response = alsc(review_text, alsc_schema, ate_out, ote_out)
    alsc_out = response['response']

    return alsc_out

In [54]:
review_text = df.loc[1, 'text']
atepc_json = df.loc[1, 'atepc_json']
review_text

'Very unfriendly staff and service !'

In [ ]:
aspects, sentiments = extract_absa_fields(atepc_json)
absa_fields = {'aspects': aspects, 'sentiments': sentiments}
result = nlp_absa_pipeline(review_text, absa_fields, ate_schema, ote_schema, alsc_schema)
print(result['response'])

### Single prompt
Try to do the absa using a single prompt

## gpt-oss:20b 

### Pipeline prompts
- aspect term extraction (ATE), which identifies the features or aspects being discussed (e.g., "food"), 
- opinion term extraction (OTE), which finds the words expressing an opinion (e.g., "delicious"), 
- aspect-level sentiment classification (ASC)

# NLP feed
feed the result of NLP as the imput of LLM